# Squad 3 — Processamento Batch — `ecommerce_rastreamento_entregas`

**Responsável:** Ygor Moraes  
**Squad:** 3 — Batch para Negócio  
**Tabela origem:** `ecommerce_rastreamento_entregas.csv`  
**Tabela destino:** `squad3.ecommerce_rastreamento_entregas`

## Objetivo

Validar o fluxo ponta a ponta solicitado para o estágio:

1. Conectar ao Azure Data Lake Storage Gen2.
2. Ler a tabela `ecommerce_rastreamento_entregas.csv`.
3. Entender o schema e realizar análise exploratória básica.
4. Gravar a tabela no Azure SQL Database no padrão `squad3.nome_da_tabela`.

## Observações de segurança

- As credenciais são lidas via Databricks Secrets.
- Nenhum segredo deve ser impresso no output do notebook.
- Antes do commit, limpar todos os outputs do notebook.

In [0]:
SECRET_SCOPE = "merca_cliente"

ADLS_CLIENT_ID = dbutils.secrets.get(scope=SECRET_SCOPE, key="adls_client_id")
ADLS_TENANT_ID = dbutils.secrets.get(scope=SECRET_SCOPE, key="adls_tenant_id")
ADLS_CLIENT_SECRET = dbutils.secrets.get(scope=SECRET_SCOPE, key="adls_client_secret")

ADLS_STORAGE_ACCOUNT_NAME = dbutils.secrets.get(scope=SECRET_SCOPE, key="adls_storage_account_name")
ADLS_CONTAINER = dbutils.secrets.get(scope=SECRET_SCOPE, key="adls_container")

SQL_HOST = dbutils.secrets.get(scope=SECRET_SCOPE, key="sql_host")
SQL_DATABASE = dbutils.secrets.get(scope=SECRET_SCOPE, key="sql_database")
SQL_USERNAME = dbutils.secrets.get(scope=SECRET_SCOPE, key="sql_username")
SQL_PASSWORD = dbutils.secrets.get(scope=SECRET_SCOPE, key="sql_password")

## Validação das secrets

In [0]:
required_secrets = [
    "adls_client_id",
    "adls_tenant_id",
    "adls_client_secret",
    "adls_storage_account_name",
    "adls_container",
    "sql_host",
    "sql_database",
    "sql_username",
    "sql_password",
]

for key in required_secrets:
    value = dbutils.secrets.get(scope=SECRET_SCOPE, key=key)
    assert value is not None and value.strip() != "", f"Secret vazia ou ausente: {key}"

print("Secrets lidas com sucesso.")

## Parâmetros do processamento

In [0]:
SOURCE_FILE = "ecommerce_rastreamento_entregas.csv"

TARGET_SCHEMA = "squad3"
TARGET_TABLE = "ecommerce_rastreamento_entregas"
TARGET_FULL_TABLE = f"{TARGET_SCHEMA}.{TARGET_TABLE}"

BASE_ADLS_PATH = f"abfss://{ADLS_CONTAINER}@{ADLS_STORAGE_ACCOUNT_NAME}.dfs.core.windows.net/"
SOURCE_PATH = f"{BASE_ADLS_PATH}{SOURCE_FILE}"

print(f"Arquivo origem: {SOURCE_FILE}")
print(f"Tabela destino: {TARGET_FULL_TABLE}")

## Opções de autenticação do ADLS

In [0]:
adls_options = {
    f"fs.azure.account.auth.type.{ADLS_STORAGE_ACCOUNT_NAME}.dfs.core.windows.net": "OAuth",
    f"fs.azure.account.oauth.provider.type.{ADLS_STORAGE_ACCOUNT_NAME}.dfs.core.windows.net": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    f"fs.azure.account.oauth2.client.id.{ADLS_STORAGE_ACCOUNT_NAME}.dfs.core.windows.net": ADLS_CLIENT_ID,
    f"fs.azure.account.oauth2.client.secret.{ADLS_STORAGE_ACCOUNT_NAME}.dfs.core.windows.net": ADLS_CLIENT_SECRET,
    f"fs.azure.account.oauth2.client.endpoint.{ADLS_STORAGE_ACCOUNT_NAME}.dfs.core.windows.net": f"https://login.microsoftonline.com/{ADLS_TENANT_ID}/oauth2/token",
}

print("Opções de autenticação do ADLS configuradas.")

## Teste de conexão com ADLS e localização do arquivo

In [0]:
from pyspark.sql.functions import col

df_arquivos = (
    spark.read
    .format("binaryFile")
    .option("recursiveFileLookup", "true")
    .options(**adls_options)
    .load(BASE_ADLS_PATH)
)

df_arquivo_rastreamento = (
    df_arquivos
    .select("path", "length", "modificationTime")
    .filter(col("path").endswith(SOURCE_FILE))
)

display(df_arquivo_rastreamento)

qtde_arquivos = df_arquivo_rastreamento.count()
assert qtde_arquivos > 0, f"Arquivo não encontrado no ADLS: {SOURCE_FILE}"

print(f"Arquivo localizado com sucesso: {SOURCE_FILE}")

## Leitura do CSV com Spark

In [0]:
df_rastreamento = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .options(**adls_options)
    .load(SOURCE_PATH)
)

print("Arquivo CSV lido com sucesso.")

## Análise exploratória básica

In [0]:
total_linhas = df_rastreamento.count()
total_colunas = len(df_rastreamento.columns)

print(f"Total de linhas: {total_linhas}")
print(f"Total de colunas: {total_colunas}")

df_rastreamento.printSchema()

## Validação estrutural da tabela

In [0]:
assert total_linhas > 0, "A tabela ecommerce_rastreamento_entregas foi lida sem registros."
assert total_colunas > 0, "A tabela ecommerce_rastreamento_entregas foi lida sem colunas."

colunas_duplicadas = [
    coluna
    for coluna in df_rastreamento.columns
    if df_rastreamento.columns.count(coluna) > 1
]

assert not colunas_duplicadas, f"Colunas duplicadas encontradas: {colunas_duplicadas}"

print("Validação estrutural concluída com sucesso.")
print("Colunas encontradas:")
print(df_rastreamento.columns)

## Amostra segura dos dados

In [0]:
sensitive_terms = ["senha", "password", "secret", "token", "email", "cpf", "cnpj"]

safe_columns = [
    c for c in df_rastreamento.columns
    if not any(term in c.lower() for term in sensitive_terms)
]

display(df_rastreamento.select(safe_columns).limit(10))

## Validações básicas de qualidade

In [0]:
from pyspark.sql.functions import countDistinct

dq_metrics = {
    "total_linhas": total_linhas,
    "total_colunas": total_colunas,
}

if "id_rastreamento" in df_rastreamento.columns:
    dq_metrics["id_rastreamento_nulos"] = df_rastreamento.filter(col("id_rastreamento").isNull()).count()
    dq_metrics["id_rastreamento_distintos"] = df_rastreamento.select(countDistinct("id_rastreamento")).collect()[0][0]

if "id_pedido" in df_rastreamento.columns:
    dq_metrics["id_pedido_nulos"] = df_rastreamento.filter(col("id_pedido").isNull()).count()
    dq_metrics["id_pedido_distintos"] = df_rastreamento.select(countDistinct("id_pedido")).collect()[0][0]

df_dq_metrics = spark.createDataFrame([dq_metrics])

display(df_dq_metrics)

## Resumo de valores nulos por coluna

In [0]:
from pyspark.sql.functions import sum as spark_sum, when

null_summary = df_rastreamento.select([
    spark_sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in df_rastreamento.columns
])

display(null_summary)

## Verificação da tabela de destino no SQL Server

In [0]:
consulta_tabela = f"""
SELECT
    TABLE_SCHEMA,
    TABLE_NAME,
    TABLE_TYPE
FROM INFORMATION_SCHEMA.TABLES
WHERE TABLE_SCHEMA = '{TARGET_SCHEMA}'
  AND TABLE_NAME = '{TARGET_TABLE}'
"""

df_tabela_existente = (
    spark.read
    .format("sqlserver")
    .option("host", SQL_HOST)
    .option("port", "1433")
    .option("database", SQL_DATABASE)
    .option("query", consulta_tabela)
    .option("user", SQL_USERNAME)
    .option("password", SQL_PASSWORD)
    .load()
)

display(df_tabela_existente)

In [0]:
tabela_ja_existe = df_tabela_existente.count() > 0

if tabela_ja_existe:
    print(f"A tabela {TARGET_FULL_TABLE} já existe no SQL Server.")
    print("Como esta tabela é responsabilidade do aluno, ela será sobrescrita na etapa de gravação.")
else:
    print(f"A tabela {TARGET_FULL_TABLE} ainda não existe no SQL Server.")
    print("Ela será criada na etapa de gravação.")

## Gravação da tabela no SQL Server

Tabela destino:

`squad3.ecommerce_rastreamento_entregas`

In [0]:
(
    df_rastreamento.write
    .format("sqlserver")
    .option("host", SQL_HOST)
    .option("port", "1433")
    .option("database", SQL_DATABASE)
    .option("dbtable", TARGET_FULL_TABLE)
    .option("user", SQL_USERNAME)
    .option("password", SQL_PASSWORD)
    .mode("overwrite")
    .save()
)

print(f"Tabela gravada com sucesso no SQL Server: {TARGET_FULL_TABLE}")

## Validação da gravação

In [0]:
df_validacao_sql = (
    spark.read
    .format("sqlserver")
    .option("host", SQL_HOST)
    .option("port", "1433")
    .option("database", SQL_DATABASE)
    .option("dbtable", TARGET_FULL_TABLE)
    .option("user", SQL_USERNAME)
    .option("password", SQL_PASSWORD)
    .load()
)

total_gravado_sql = df_validacao_sql.count()

print(f"Total de registros na origem: {total_linhas}")
print(f"Total de registros gravados em {TARGET_FULL_TABLE}: {total_gravado_sql}")

assert total_gravado_sql == total_linhas, "Divergência entre quantidade de registros na origem e no SQL Server."

display(df_validacao_sql.select(safe_columns).limit(10))